# ADMET feature-Ising QSVT: experiment and API handoff

Run the canonical molecular pipeline through a reproducible runtime package:
Morgan counts → information-screened interactions → molecule-specific Ising
Hamiltonian → variational AGP → heralded QSVT evolution.

The default uses public structures with invented labels. It tests implementation
and numerical consistency; it supplies no ADMET prediction benchmark. The two
companion derivation notebooks expose the chemical, statistical, and Hamiltonian
construction steps. Setup and validation scope are in the project README.

The operators encode information from molecular features in chosen dimensionless
energy units (ℏ = 1). They are not an electronic-structure Hamiltonian or a
calibrated physical model of an ADMET mechanism.


In [ ]:
from pathlib import Path
import json, os, sys
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src' / 'admet_submission_entry.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run from the ADMET project directory or its pipeline directory')
sys.path[:0] = [str(PROJECT_ROOT/'src'), str(PROJECT_ROOT/'python_classes')]
import admet_submission_entry as entry
from admet_curation import FIXTURE_NAME
from runtime_input_protocol import decode_runtime_input_values
from triple_alpha_client import TripleAlphaClient


## 1. Reproducible molecular instance

Keep every retained interaction. The curation notebook defines the count
fingerprints, discrete MI ranking, KSG conditional screening, signed
co-information, and the compact mapping from fingerprint indices to qubits.
Here all three workflows bind the same molecule's scaled counts to local fields.
The fit scope is the supplied exploratory table, without a held-out prediction claim.

In [ ]:
DATA_MODE = 'fixture'
DATASET = FIXTURE_NAME
SAMPLE_INDEX = 13
INTERPOLATION = 0.5
LAMBDA_DOT = 1.0
PHYSICAL_TIME = 0.1
EPSILON = 1e-6
OUTPUT_DIR = Path(os.environ.get('ADMET_OUTPUT_DIR', PROJECT_ROOT/'artifacts'/'runtime'))
payload = entry.build_molecular_payload(
    data_mode=DATA_MODE, dataset=DATASET, sample_index=SAMPLE_INDEX,
    interpolation=INTERPOLATION, lambda_dot=LAMBDA_DOT)
validator = entry.build_validator(payload)
print(json.dumps({'dataset': payload['dataset'], 'sample': SAMPLE_INDEX,
                  'system_qubits': validator.num_system_qubits,
                  'pauli_terms': validator.num_terms,
                  'feature_vector': payload['feature_vector'],
                  'provenance': payload['curation_provenance']}, indent=2))


## 2. Counterdiabatic snapshot

The driver is $H_0=-\sum_iX_i$, the molecular problem is
$H_p(\theta)=H_A+\sum_i\theta_iZ_i$, and
$H(s)=(1-s)H_0+sH_p$. With $O_0=H_p-H_0$,
$O_1=[H,O_0]$, and $O_2=[H,O_1]$, the first-order ansatz is

$$A_s^{(1)}=i\alpha_1O_1,\qquad
\alpha_1=-\|O_1\|_{\rm HS}^2/\|O_2\|_{\rm HS}^2,\qquad
H_{\rm CD}=H(s)+\dot{s}A_s^{(1)}.$$

The CD notebook checks the merged Pauli algebra and variational stationarity
against independent dense matrices. This run evolves the **frozen** snapshot
$\exp(-itH_{\rm CD})$; it does not integrate an annealing schedule.

In [ ]:
print(json.dumps(payload['agp_metadata'], indent=2))
assert payload['agp_metadata']['interaction_truncation']['dropped_count'] == 0
assert payload['agp_metadata']['ansatz_order'] == 1


## 3. LCU, qubitization, and both QSVT branches

Let $\lambda=\sum_j|c_j|$, $P|0\rangle=\sum_j\sqrt{|c_j|/\lambda}|j\rangle$,
and $B=P^\dagger\operatorname{SELECT}P$. The zero-index block is
$H_{\rm CD}/\lambda$, and $W=(2\Pi_0-I)B$ is the qubitized walk.
A degree-$d$ projector-phase sequence uses $d$ block queries and $d+1$ phases.

Even cosine and odd sine sequences are both required. Coherent $U/U^\dagger$
averaging extracts their real polynomials. A second ancilla combines them as
$C-iS$ with scale-aware weights. Successful zero measurements of the branch,
extraction, and index ancillas encode
$\beta\exp(-itH_{\rm CD})$, with $\beta\approx1/4$ and success probability
approximately $1/16$. This is heralded evolution, without amplitude amplification.

The numerical error bound includes the Jacobi–Anger tail, reconstructed phase
response, and a float64 allowance. It is not an interval-arithmetic proof.

In [ ]:
bundle_dir, qsvt = entry.prepare_runtime_inputs(
    payload, OUTPUT_DIR, physical_time=PHYSICAL_TIME, epsilon=EPSILON)
reference = qsvt['mathematical_reference']
assert reference['recovered_evolution_error'] <= qsvt['recovered_evolution_error_bound'] + 1e-11
assert qsvt['recovered_evolution_error_bound'] <= EPSILON
print(json.dumps({
    'lambda': qsvt['lambda_value'], 'tau': qsvt['tau'],
    'cosine_degree': qsvt['cosine']['polynomial_degree'],
    'sine_degree': qsvt['sine']['polynomial_degree'],
    'success_probability_lower_bound': qsvt['success_probability_lower_bound'],
    'reference': reference, 'logical_resources': qsvt['logical_resources']}, indent=2))


## 4. Strict runtime protocol and reproducible package

Protocol v2 contains five dimensions, molecular feature values, signed Pauli
rows, both phase sequences, and the branch rotation. Features initialize the
system with RY(πθ). The decoder checks exact lengths, integer dimensions, codes,
finite values, and register capacity independently of the packer.

The package records strict JSON, checksums, data identity, settings, environment,
and source hashes. Candidate QBE source remains separate from generated output.
Small gate-level reference systems are tested against direct matrix exponentiation;
this four-qubit instance compares every Hamiltonian eigenvalue and direct expm
without allocating the much larger ancilla unitary.

Register counts cover system, index, branch, and extraction qubits. Temporary
control workspace, native depth, allocation/reuse, and physical backend costs
require compiler validation and are not inferred from these counts.


In [ ]:
manifest = entry.verify_runtime_bundle(bundle_dir)
runtime_doc = qsvt['synchronization']['runtime_document']
decoded = decode_runtime_input_values(runtime_doc['inputs'])
assert decoded['num_terms'] == validator.num_terms
assert qsvt['synchronization']['status'] == 'local_contract_verified'
assert not manifest['live_api_execution_allowed']
print(json.dumps({'bundle': str(bundle_dir), 'protocol': runtime_doc['protocol_version'],
                  'inputs': runtime_doc['input_count'],
                  'total_logical_qubits': runtime_doc['num_total_qubits'],
                  'input_sha256': runtime_doc['inputs_sha256'],
                  'qbe_modules_hashed': len(manifest['module_hashes'])}, indent=2))


## 5. API boundary and remaining external validation

No credentials or requests are needed for this notebook. Mock tests cover
create → awaiting input → update → completion → results, plus failures, timeouts,
redirection rejection, malformed responses, and credential redaction.

Native Beryllium gate semantics, compilation, deployed program identity, and the
service's HTTP contract remain unverified. Protocol v2 changes the circuit and
requires a fresh verified deployment. The adapter blocks execution until those
checks and explicit live opt-in are supplied. No hardware result is implied.

In [ ]:
assert manifest['compiler_verified'] is False
assert manifest['gate_convention_verified'] is False
assert manifest['deployment_verified'] is False
assert manifest['service_contract_verified'] is False
# Reserved invalid domain and placeholder are used only to demonstrate preflight.
client = TripleAlphaClient('https://example.invalid', 'offline-placeholder')
try:
    client.execute(runtime_doc['inputs'])
except RuntimeError as exc:
    assert 'Live execution is disabled' in str(exc)
    print('API preflight blocked as intended; no request was sent.')
else:
    raise AssertionError('Offline preflight unexpectedly allowed execution')
